# Force Field Typification with TFSI

**"How do I assign force field parameters to my molecules?"**

This guide demonstrates MolPy's **Typifier** system, which automatically assigns force field types and parameters to molecular structures. We'll use **TFSI** (bis(trifluoromethylsulfonyl)imide) as a concrete example—a common ionic liquid anion.

## What is a Typifier?

A **Typifier** maps atoms, bonds, angles, and dihedrals in your molecular structure to specific types defined in a force field. It examines the chemical environment (element, connectivity, bonding patterns) and assigns:

- **Atom types**: Define nonbonded interactions (charge, σ, ε)
- **Bond types**: Harmonic bond parameters (k, r₀)
- **Angle types**: Harmonic angle parameters (k, θ₀)
- **Dihedral types**: Torsional parameters (k, n, δ)

## Why TFSI?

TFSI is an excellent example because it contains:
- **Multiple elements**: C, F, S, N, O
- **Diverse bonding**: Single bonds, S=O double bonds, S-N bonds
- **Symmetry**: Two CF₃SO₂ groups connected by nitrogen
- **Challenging chemistry**: Sulfonyl groups with partial charges

TFSI structure: `F₃C-SO₂-N⁻-SO₂-CF₃`

## Setup: Imports

In [ ]:
import molpy as mp
from molpy.core.atomistic import Atomistic, Angle, Dihedral, Atom
from molpy.typifier.atomistic import OplsAtomisticTypifier
from molpy.parser.smiles import parse_smiles, smilesir_to_atomistic
from molpy.adapter.rdkit import RDKitAdapter
from molpy.compute.rdkit import Generate3D


## Step 1: Build TFSI Molecule

First, we'll create the TFSI structure from SMILES and generate 3D coordinates.

In [ ]:
# TFSI SMILES: bis(trifluoromethylsulfonyl)imide anion
tfsi_smiles = "C(F)(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F"

# Parse SMILES using molpy's parser
ir = parse_smiles(tfsi_smiles)

# Convert SmilesGraphIR to Atomistic using molpy's converter function
tfsi = smilesir_to_atomistic(ir)

# Generate 3D coordinates using RDKit (only step using RDKit)
adapter = RDKitAdapter(internal=tfsi)
generate_3d = Generate3D(add_hydrogens=False, embed=True, optimize=True, update_internal=True)
adapter = generate_3d(adapter)
tfsi = adapter.get_internal()

# Generate topology (bonds, angles, dihedrals)
tfsi.get_topo(gen_angle=True, gen_dihe=True)

print(f"✅ Built TFSI molecule:")
print(f"   Atoms: {len(tfsi.atoms)}")
print(f"   Bonds: {len(tfsi.bonds)}")
print(f"   Angles: {len(tfsi.links.bucket(Angle))}")
print(f"   Dihedrals: {len(tfsi.links.bucket(Dihedral))}")

# Show atom composition
from collections import Counter
elements = Counter(atom.get("symbol", "?") for atom in tfsi.atoms)
print(f"\n   Composition: {dict(elements)}")


✅ Built TFSI molecule:
   Atoms: 15
   Bonds: 14
   Angles: 25
   Dihedrals: 24

   Composition: {'C': 2, 'F': 6, 'S': 2, 'O': 4, 'N': 1}


## Step 2: Load Force Field

MolPy supports OPLS-AA and other force fields via XML format.

**Note**: Standard OPLS-AA does not include parameters for TFSI (ionic liquids require specialized force fields). For this demonstration, we'll use the standard OPLS-AA force field and set `strict_typing=False` to allow partial typing. In production use, you would need a force field file that includes TFSI-specific atom types.


In [ ]:
# Load OPLS-AA force field
# Note: You may need a specialized force field for TFSI
# This example assumes you have oplsaa.xml with TFSI parameters
forcefield_path = "gaff.xml"
ff = mp.io.read_xml_forcefield(forcefield_path)

print(f"✅ Loaded force field: {forcefield_path}")
print(f"   Atom types: {len(list(ff.get_types(mp.AtomType)))}")
print(f"   Bond types: {len(list(ff.get_types(mp.BondType)))}")
print(f"   Angle types: {len(list(ff.get_types(mp.AngleType)))}")
print(f"   Dihedral types: {len(list(ff.get_types(mp.DihedralType)))}")


✅ Loaded force field: gaff.xml
   Atom types: 71
   Bond types: 652
   Angle types: 3114
   Dihedral types: 0


## Step 3: Create Typifier

The `OplsAtomisticTypifier` is a comprehensive typifier that assigns all types in one call:

- **Atom typing**: Uses SMARTS pattern matching to identify chemical environments
- **Pair typing**: Assigns nonbonded parameters (charge, σ, ε)
- **Bond typing**: Matches bond types based on connected atom types
- **Angle typing**: Matches angle types based on three-atom sequences
- **Dihedral typing**: Matches torsion types based on four-atom sequences

In [ ]:
# Create typifier
# Note: TFSI requires specialized force field parameters not in standard OPLS-AA
# Setting strict_typing=False allows the demo to continue with partial typing
typifier = OplsAtomisticTypifier(
    forcefield=ff,
    skip_atom_typing=False,  # Assign atom types
    skip_pair_typing=False,  # Assign nonbonded parameters
    skip_bond_typing=False,  # Assign bond parameters
    skip_angle_typing=False, # Assign angle parameters
    skip_dihedral_typing=False, # Assign dihedral parameters
    strict_typing=False  # Allow untyped atoms (TFSI needs specialized FF parameters)
)

print("✅ Created OplsAtomisticTypifier")
print("   This will assign all types (atoms, bonds, angles, dihedrals)")


✅ Created OplsAtomisticTypifier
   This will assign all types (atoms, bonds, angles, dihedrals)


/workspaces/molcrafts/molpy/src/molpy/typifier/atomistic.py:397: UserWarning: Failed to parse SMARTS for gaff_sy: [S;X4](=*)-*#*, error: No terminal matches '=' in the current parser context, at line 1 col 8

[S;X4](=*)-*#*
       ^
Expected one of: 
	* SYMBOL
	* STAR
	* LSQB

  self.pattern_dict = self._extract_patterns()
/workspaces/molcrafts/molpy/src/molpy/typifier/atomistic.py:397: UserWarning: Failed to parse SMARTS for gaff_h1: [#1;X1][C](=[#6])[F,Cl,Br,I], error: No terminal matches '=' in the current parser context, at line 1 col 12

[#1;X1][C](=[#6])[F,Cl,Br,I]
           ^
Expected one of: 
	* SYMBOL
	* STAR
	* LSQB

  self.pattern_dict = self._extract_patterns()
/workspaces/molcrafts/molpy/src/molpy/typifier/atomistic.py:397: UserWarning: Failed to parse SMARTS for gaff_nb: [#7;X2;r5](:[#6])[#6], error: No terminal matches ':' in the current parser context, at line 1 col 12

[#7;X2;r5](:[#6])[#6]
           ^
Expected one of: 
	* SYMBOL
	* STAR
	* LSQB

  self.pattern_dict 

## Step 4: Typify the Molecule

Now we apply the typifier to assign all force field parameters to TFSI.

In [ ]:
# Typify TFSI
tfsi_typed = typifier.typify(tfsi)

print("✅ Typified TFSI molecule")
print("\n📋 Sample atom types:")

# Show first 5 atoms with their types
for i, atom in enumerate(tfsi_typed.atoms[:5]):
    symbol = atom.get("symbol", "?")
    atom_type = atom.get("type", "untyped")
    charge = atom.get("charge", 0.0)
    print(f"   Atom {i+1}: {symbol:2s} → type={atom_type:10s} q={charge:7.4f}")

print("\n📋 Sample bond types:")
for i, bond in enumerate(tfsi_typed.bonds[:3]):
    bond_type = bond.get("type", "untyped")
    k = bond.get("k", 0.0)
    r0 = bond.get("length", 0.0)
    i_sym = bond.itom.get("symbol", "?")
    j_sym = bond.jtom.get("symbol", "?")
    print(f"   Bond {i+1}: {i_sym}-{j_sym} → type={bond_type:15s} k={k:8.2f} r0={r0:.4f}")

ValueError: No pair type found for atom type: gaff_f

## Step 5: Inspect Typification Results

Let's examine the assigned types in detail to understand how the typifier works.

In [ ]:
# Collect all unique atom types
atom_types = set()
for atom in tfsi_typed.atoms:
    atom_type = atom.get("type")
    if atom_type:
        atom_types.add(atom_type)

print(f"📊 Unique atom types in TFSI: {len(atom_types)}")
for atype in sorted(atom_types):
    # Count atoms of this type
    count = sum(1 for a in tfsi_typed.atoms if a.get("type") == atype)
    # Get example atom
    example = next(a for a in tfsi_typed.atoms if a.get("type") == atype)
    symbol = example.get("symbol", "?")
    charge = example.get("charge", 0.0)
    print(f"   {atype:15s}: {count:2d}× {symbol:2s} (q={charge:7.4f})")

# Show charge neutrality
total_charge = sum(atom.get("charge", 0.0) for atom in tfsi_typed.atoms)
print(f"\n⚡ Total molecular charge: {total_charge:.6f}")
print(f"   (TFSI is an anion, expected: -1.0)")

## Step 6: Export to LAMMPS

Once typified, we can export the molecule to LAMMPS format for molecular dynamics simulations.

In [ ]:
from pathlib import Path
from molpy.io.data.lammps import LammpsDataWriter
from molpy.io.forcefield.lammps import LAMMPSForceFieldWriter
from molpy.core.box import Box

# Create output directory
output_dir = Path("user-guide-output") / "07_typifier_tfsi"
output_dir.mkdir(parents=True, exist_ok=True)

# Convert to Frame for export
tfsi_frame = tfsi_typed.to_frame()

# Add a simulation box (required for LAMMPS)
box = Box.cubic(length=30.0)  # 30 Å cubic box
tfsi_frame.metadata["box"] = box

# Collect all types for force field export
def collect_types(frame):
    atom_types = set()
    bond_types = set()
    angle_types = set()
    dihedral_types = set()
    
    if "atoms" in frame and "type" in frame["atoms"]:
        for t in frame["atoms"]["type"]:
            if t and not str(t).isdigit():
                atom_types.add(str(t))
    
    if "bonds" in frame and "type" in frame["bonds"]:
        for t in frame["bonds"]["type"]:
            if t and not str(t).isdigit():
                bond_types.add(str(t))
    
    if "angles" in frame and "type" in frame["angles"]:
        for t in frame["angles"]["type"]:
            if t and not str(t).isdigit():
                angle_types.add(str(t))
    
    if "dihedrals" in frame and "type" in frame["dihedrals"]:
        for t in frame["dihedrals"]["type"]:
            if t and not str(t).isdigit():
                dihedral_types.add(str(t))
    
    return atom_types, bond_types, angle_types, dihedral_types

atom_types, bond_types, angle_types, dihedral_types = collect_types(tfsi_frame)

# Write force field file
ff_path = output_dir / "tfsi.ff"
ff_writer = LAMMPSForceFieldWriter(ff_path)
ff_writer.write(
    ff,
    atom_types=atom_types,
    bond_types=bond_types,
    angle_types=angle_types,
    dihedral_types=dihedral_types,
)

# Write data file
data_path = output_dir / "tfsi.data"
data_writer = LammpsDataWriter(data_path, atom_style="full")
data_writer.write(tfsi_frame)

print(f"✅ Exported LAMMPS files:")
print(f"   Force field: {ff_path}")
print(f"   Data file: {data_path}")
print(f"\n   Atom types: {len(atom_types)}")
print(f"   Bond types: {len(bond_types)}")
print(f"   Angle types: {len(angle_types)}")
print(f"   Dihedral types: {len(dihedral_types)}")

## Understanding the Typifier

### How Atom Typing Works

The `OplsAtomTypifier` uses **SMARTS pattern matching** to identify atom types:

1. **Pattern extraction**: Reads SMARTS patterns from force field XML (`def` attribute)
2. **Priority resolution**: Handles overlapping patterns using `priority` and `overrides` attributes
3. **Layered matching**: Processes patterns in layers to resolve type dependencies
4. **Chemical environment**: Matches based on element, hybridization, neighbors, bonds

For TFSI, typical atom types might include:
- **Fluorine**: `opls_f_tfsi` - Fluorine in CF₃ group
- **Carbon**: `opls_c_tfsi` - Carbon in CF₃ group
- **Sulfur**: `opls_s_tfsi` - Sulfur in SO₂ group
- **Oxygen**: `opls_o_tfsi` - Oxygen in SO₂ group
- **Nitrogen**: `opls_n_tfsi` - Central nitrogen

### How Bond/Angle/Dihedral Typing Works

After atom typing, bonded interactions are typed by:

1. **Bond typing**: Match based on atom types at both ends
2. **Angle typing**: Match based on three consecutive atom types
3. **Dihedral typing**: Match based on four consecutive atom types

The typifier supports:
- **Wildcards**: `*` matches any type
- **Classes**: Group similar types together
- **Bidirectional matching**: A-B matches B-A for bonds

### Strict vs. Non-Strict Mode

- **Strict mode** (`strict_typing=True`): Raises error if any atom cannot be typed
- **Non-strict mode** (`strict_typing=False`): Silently skips untyped atoms

Use strict mode during development to catch missing force field parameters!

## Summary

You now understand how to use MolPy's typifier system:

1. ✅ **Build molecule**: Create structure from SMILES or other formats
2. ✅ **Load force field**: Read force field XML with type definitions
3. ✅ **Create typifier**: Initialize `OplsAtomisticTypifier` with desired options
4. ✅ **Typify**: Assign all types and parameters in one call
5. ✅ **Inspect**: Examine assigned types and parameters
6. ✅ **Export**: Write to LAMMPS or other simulation formats

### Key Takeaways

- **Typifier separates structure from parameters**: Build molecules first, assign physics later
- **SMARTS-based matching**: Flexible, chemical-environment-aware atom typing
- **Comprehensive**: Handles atoms, bonds, angles, dihedrals in one call
- **Strict mode**: Catch missing parameters during development

### Next Steps

- **Custom force fields**: Create your own XML force field definitions
- **Mixed systems**: Typify polymers, solvents, and ions together
- **Validation**: Compare assigned parameters with literature values
- **Simulation**: Use typified structures in LAMMPS, GROMACS, etc.